# Project 3 Part B

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
HF_TOKEN = "YOUR_HF_TOKEN_HERE" 
login(HF_TOKEN)

In [2]:
!pip install -q transformers accelerate pydantic torch outlines
!pip install -q trl peft accelerate bitsandbytes
!pip install -q transformers datasets
!pip -q install llguidance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 20.0 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
import random
import os
import re
import json
import math
import torch
import gc
from datetime import datetime
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

# Model helper
def load_model(model_name, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )

    if lora_path:
        model = PeftModel.from_pretrained(model, lora_path)

    model.eval()
    return model, tokenizer

def cleanup(*objects):
    """Delete model/tokenizer objects and free GPU memory."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [4]:
# Consts
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = (
    "You are a careful math solver. Think about the following problem step by step.\n"
    "At the end, output exactly ONE line with the final answer in LaTeX, strictly follows the format:\n"
    "\\boxed{<integer>}\n"
    "Please only include integer in the boxed"
    "Do not include any other text after the boxed answer. \n"
    "Do not generate any boxed content other than the final answer. \n"
)

In [5]:
#load model and dataset
from datasets import load_dataset
test_ds = load_dataset("gsm8k", "main")["test"]
model, tokenizer = load_model(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Task 1 Dataset Inspection and Sanity Checks

### Task 1.1 Load and Inspect JSONL Files

In [8]:
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/share_data")
TABLE_DIR = DATA_DIR / "da-dev-tables"


q_path = DATA_DIR / "da-dev-questions.jsonl"
l_path = DATA_DIR / "da-dev-labels.jsonl"

def load_jsonl(path: Path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

questions = load_jsonl(q_path)
labels = load_jsonl(l_path)

# ====== Q14 outputs ======
print("Number of questions:", len(questions))
print("Number of labels:", len(labels))

print("\n--- Question record keys ---")
print(sorted(list(questions[0].keys())))
print("\n--- One example question record ---")
print(json.dumps(questions[0], indent=2, ensure_ascii=False)[:2000])  # truncate for readability

print("\n--- Label record keys ---")
print(sorted(list(labels[0].keys())))
print("\n--- One example label record ---")
print(json.dumps(labels[0], indent=2, ensure_ascii=False)[:2000])     # truncate for readability

Number of questions: 257
Number of labels: 257

--- Question record keys ---
['concepts', 'constraints', 'file_name', 'format', 'id', 'level', 'question']

--- One example question record ---
{
  "id": 0,
  "question": "Calculate the mean fare paid by the passengers.",
  "concepts": [
    "Summary Statistics"
  ],
  "constraints": "Calculate the mean fare using Python's built-in statistics module or appropriate statistical method in pandas. Rounding off the answer to two decimal places.",
  "format": "@mean_fare[mean_fare_value] where \"mean_fare_value\" is a floating-point number rounded to two decimal places.",
  "file_name": "test_ave.csv",
  "level": "easy"
}

--- Label record keys ---
['common_answers', 'id']

--- One example label record ---
{
  "id": 0,
  "common_answers": [
    [
      "mean_fare",
      "34.65"
    ]
  ]
}


#### Question 14

The dataset contains 257 questions and 257 labels, indicating a one-to-one correspondence between questions and their labels.

Each question record contains the following keys: `concepts`, `constraints`, `file_name`, `format`, `id`, `level`, and `question`. An example question asks to calculate the mean fare paid by passengers, specifies summary statistics as the concept, includes rounding constraints, defines the expected output format, references a CSV file, and is labeled as easy.

Each label record contains the keys `id` and `common_answers`. The `id` corresponds to the question ID, and `common_answers` provides acceptable answers for evaluation.

### Task 1.2 Inspect the Table References

#### Question 15

In [9]:
import random
import pandas as pd

# Select 3 random question IDs
random_ids = random.sample(range(len(questions)), 3)

results = []

for qid in random_ids:
    q = questions[qid]
    csv_file = q["file_name"]
    csv_path = TABLE_DIR / csv_file
    
    print("="*70)
    print(f"Question ID: {qid}")
    print(f"Referenced CSV: {csv_file}")
    
    df = pd.read_csv(csv_path)
    
    print("\nShape:", df.shape)
    print("\nDtypes:\n", df.dtypes)
    print("\nHead (3 rows):\n", df.head(3))
    
    print("\nQuestion:")
    print(q["question"])
    
    results.append((qid, csv_file, df.shape, list(df.columns)))

Question ID: 182
Referenced CSV: abalone.csv

Shape: (4177, 9)

Dtypes:
 Sex                   str
Length            float64
Diameter          float64
Height            float64
Whole weight      float64
Shucked weight    float64
Viscera weight    float64
Shell weight      float64
Rings               int64
dtype: object

Head (3 rows):
   Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   

   Shell weight  Rings  
0          0.15     15  
1          0.07      7  
2          0.21      9  

Question:
Perform comprehensive data preprocessing on the abalone dataset. Handle any missing values and scale the variables (length, diameter, height, whole weight, shucked weight, viscera weight, shell weight) using min-max normalizat

### Task 1.3 Understand the Required Answer Format

#### Question 16

Two examples where the required format contains multiple answer slots are **Question ID 275** and **Question ID 144**.  
For **ID 275**, the required format has **four** slots: `@duplicate_count[...]`, `@usflux_mean[...]`, `@new_feature_mean[...]`, and `@model_accuracy[...]`. For **ID 144**, the format has **six** slots: `@mean_dem[...]`, `@mean_gop[...]`, `@std_dev_dem[...]`, `@std_dev_gop[...]`, `@dist_dem[...]`, and `@dist_gop[...]`.

**How the dataset represents multi-part answers in the labels:** multi-part answers are stored in each label record’s `common_answers` as **multiple acceptable answer variants**, where each variant consists of **(slot_name, value)** pairs (e.g., for ID 275 there are 2 variants; for ID 144 there are 4 variants). This structure allows multiple valid completions while still keeping each required field explicitly named.

**How to evaluate such answers automatically:** parse the model output to extract all `@slot[value]` pairs, then (1) verify that the set of slots matches the required slots from the question’s `format`, (2) normalize values (strip whitespace; parse numbers; enforce rounding/tolerance rules; canonicalize strings like `"Normal"`/`"Not Normal"`), and (3) compare the resulting slot→value mapping against each acceptable variant in `common_answers`. Mark the answer correct if it matches **any** variant, and optionally allow order-insensitivity since slots are explicitly named.

In [10]:
import re
import random
import json

slot_pat = re.compile(r'@([A-Za-z0-9_]+)\[')

# Build id->record maps (safe even if ids aren't 0..N-1 or not aligned)
q_by_id = {q["id"]: q for q in questions}
l_by_id = {l["id"]: l for l in labels}

multi = []
for q in questions:
    slots = slot_pat.findall(q["format"])
    if len(slots) >= 2:
        multi.append((q["id"], slots))

print("Num questions with >=2 answer slots:", len(multi))

# pick 2 examples
example_ids = [qid for qid, _ in random.sample(multi, 2)]

for qid in example_ids:
    q = q_by_id[qid]
    lab = l_by_id[qid]
    slots = slot_pat.findall(q["format"])

    print("=" * 80)
    print("Question ID:", qid)
    print("Question:", q["question"])
    print("Format:", q["format"])
    print("Slots (in order):", slots)

    ca = lab["common_answers"]
    print("\nLabel common_answers: num variants =", len(ca))
    print("First 2 variants (truncated):")
    print(json.dumps(ca[:2], indent=2, ensure_ascii=False)[:1500])

Num questions with >=2 answer slots: 145
Question ID: 309
Question: Perform distribution analysis on the age and fare variables separately, then calculate and compare the skewness and kurtosis values for each. Additionally, count the number of values within one standard deviation from the mean, for both age and fare.
Format: @age_skewness[skewness_value]   
@age_kurtosis[kurtosis_value] 
@age_values_within_one_stdev[number]
@fare_skewness[skewness_value] 
@fare_kurtosis[kurtosis_value] 
@fare_values_within_one_stdev[number]
where "skewness_value", "kurtosis_value" are floats with two decimals, "number" is a positive integer.
Slots (in order): ['age_skewness', 'age_kurtosis', 'age_values_within_one_stdev', 'fare_skewness', 'fare_kurtosis', 'fare_values_within_one_stdev']

Label common_answers: num variants = 6
First 2 variants (truncated):
[
  [
    "fare_kurtosis",
    "33.20"
  ],
  [
    "age_values_within_one_stdev",
    "516"
  ]
]
Question ID: 655
Question: 1. Perform a correlatio

### Task 1.4 Checking the subset

#### Question 17

The selected subset of question IDs is:
[0, 5, 9, 10, 14, 18, 24, 25, 26, 55].

All selected tasks were printed and verified. Each ID exists in the dataset and references a valid CSV file. The tasks primarily involve summary statistics (mean, standard deviation), correlation analysis, distribution testing (normality checks), and basic feature engineering.

Most selected questions are labeled as *easy*, with only two marked as *medium*. The required answer formats are clearly structured with one or a small number of answer slots, and the computations involve straightforward statistical operations rather than complex multi-step modeling or advanced machine learning.

Overall, this subset consists of well-defined, numerically grounded tasks that are feasible for the model to solve, justifying their selection as solvable sub-tasks.

In [11]:
SELECTED_IDS = [0, 5, 9, 10, 14, 18, 24, 25, 26, 55]

# Build id -> question mapping (safe)
q_by_id = {q["id"]: q for q in questions}

print("Checking selected tasks:\n")

for qid in SELECTED_IDS:
    print("=" * 80)
    print(f"Question ID: {qid}")
    
    if qid in q_by_id:
        q = q_by_id[qid]
        print("Level:", q["level"])
        print("Concepts:", q["concepts"])
        print("File:", q["file_name"])
        print("Format:", q["format"])
        print("\nQuestion:")
        print(q["question"])
    else:
        print("ID not found in dataset.")

Checking selected tasks:

Question ID: 0
Level: easy
Concepts: ['Summary Statistics']
File: test_ave.csv
Format: @mean_fare[mean_fare_value] where "mean_fare_value" is a floating-point number rounded to two decimal places.

Question:
Calculate the mean fare paid by the passengers.
Question ID: 5
Level: medium
Concepts: ['Feature Engineering', 'Correlation Analysis']
File: test_ave.csv
Format: @correlation_coefficient[r_value]
where "r_value" is the Pearson correlation coefficient between 'FamilySize' and 'Fare', a number between -1 and 1, rounded to two decimal places.

Question:
Generate a new feature called "FamilySize" by summing the "SibSp" and "Parch" columns. Then, calculate the Pearson correlation coefficient (r) between the "FamilySize" and "Fare" columns.
Question ID: 9
Level: easy
Concepts: ['Summary Statistics']
File: GODREJIND.csv
Format: @mean_close_price[mean_value], where "mean_value" is a float number rounded to two decimal places. This value should be between the highe

## Task 2 Model Loading and Structured Output: Make the Planner Parseable

In [19]:
import json
import torch
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import outlines
import importlib

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=dtype,
)
planner_model = outlines.models.Transformers(hf_model, tokenizer)

class PlannerOutput(BaseModel):
    thought: str
    is_done: bool
    response: str

PLANNER_SYSTEM = """You are the PLANNER for a data-mining agent.
Output ONLY ONE JSON object with EXACT keys: thought, is_done, response.
No other text.

Rules:
- STRICT JSON using double quotes only.
- No markdown, no extra text.
- End with a closing brace: }
Template:
{"thought":"...","is_done":false,"response":"..."}
"""

# Minimal JSON Schema compatible with Outlines-core regex compiler
minimal_schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["thought", "is_done", "response"],
    "properties": {
        "thought": {"type": "string"},
        "is_done": {"type": "boolean"},
        "response": {"type": "string"},
    },
}
schema_str = json.dumps(minimal_schema)

og = importlib.import_module("outlines.generator")
term = og.JsonSchema(schema_str)

planner_gen = og.Generator(planner_model, term, None)

def run_planner(user_prompt: str, max_new_tokens: int = 512) -> str:
    prompt = f"{PLANNER_SYSTEM}\n\nUSER:\n{user_prompt}"
    return planner_gen(prompt, max_new_tokens=max_new_tokens)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

#### Question 18

In [21]:
prompts = [
    "We need the mean of column Fare from test_ave.csv. What should we do next?",
    "The CSV has SibSp, Parch, Fare. Plan next step to compute correlation between FamilySize=(SibSp+Parch) and Fare.",
    "We already computed mean_fare = 34.65. Provide the final formatted answer exactly as @mean_fare[mean_fare_value].",
    "We need to determine if 'Total Traded Quantity' is normally distributed. We have not run tests yet. What should we do next?",
    "We ran Shapiro and got p=0.001 (<0.05). Under the rule p<0.05 ⇒ not normal. Output the final answer as no."
]

objs = []
for i, p in enumerate(prompts, 1):
    j = run_planner(p, max_new_tokens=512)
    obj = PlannerOutput.model_validate_json(j)  # no try/except
    objs.append(obj)
    print(f"\n--- Prompt {i} ---")
    print(j)
    print("Parsed:", obj)

print("\nAny is_done=true?", any(o.is_done for o in objs))


--- Prompt 1 ---
{"thought":"The next step is to load the test_ave.csv file and extract the 'Fare' column to compute its mean.","is_done":false,"response":"Load test_ave.csv and calculate the mean of the 'Fare' column."}
Parsed: thought="The next step is to load the test_ave.csv file and extract the 'Fare' column to compute its mean." is_done=False response="Load test_ave.csv and calculate the mean of the 'Fare' column."

--- Prompt 2 ---
{"thought":"Calculate FamilySize by adding SibSp and Parch, then compute the correlation between FamilySize and Fare.","is_done":false,"response":"Next step: Create a new column FamilySize = SibSp + Parch, then compute the correlation between FamilySize and Fare using the CSV data."}
Parsed: thought='Calculate FamilySize by adding SibSp and Parch, then compute the correlation between FamilySize and Fare.' is_done=False response='Next step: Create a new column FamilySize = SibSp + Parch, then compute the correlation between FamilySize and Fare using t

#### Question 19
In Task 2, I constrain the Planner to output a single JSON object that matches a fixed schema (thought, is_done, response). This makes the Planner’s output machine-readable and eliminates brittle string parsing, so the agent loop can deterministically decide whether to continue (is_done=false) or stop with a final answer (is_done=true). Using schema-constrained decoding (via Outlines) also prevents malformed outputs (missing keys, extra text, invalid JSON), which is important because later stages (Coder/Executor/Observer) depend on the Planner’s output to trigger the correct next action. Overall, structured output improves reliability, reduces error-handling complexity, and makes the multi-step ReAct workflow reproducible and easier to evaluate automatically.

## Task 3 Build a ReAct Data Analysis Agent